# 6.5.6 Synthetic Data and Self-Improvement

Data augmentation (feature noise, Mixup) and self-training loop that adds pseudo-labeled examples each round.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
n, d = 100, 8
X = rng.standard_normal((n, d))
y = (X[:,0]+X[:,1]>0).astype(float)
X_aug = X + rng.normal(0, 0.15, X.shape)
lam = rng.beta(0.3, 0.3); idx = rng.permutation(n)
X_mix = lam*X + (1-lam)*X[idx]; y_mix = lam*y + (1-lam)*y[idx]
print(f'Original: {X.shape}, Aug: {X_aug.shape}, Mixup: {X_mix.shape}')

In [ ]:
# Self-training simulation
X_lab, y_lab, X_unlab = X[:20], y[:20], X[20:]
history = [len(X_lab)]
accs = [0.65]
for r in range(5):
    conf = rng.beta(2+r, max(1, 3-r), len(X_unlab))
    mask = conf >= 0.8
    X_lab = np.vstack([X_lab, X_unlab[mask]])
    y_lab = np.concatenate([y_lab, rng.integers(0,2,mask.sum())])
    X_unlab = X_unlab[~mask]
    acc = min(0.95, 0.65 + 0.07*np.log1p(r+1))
    history.append(len(X_lab)); accs.append(acc)
    print(f'Round {r+1}: labeled={len(X_lab)}, acc≈{acc:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X[:,0], 20, alpha=0.6, color='steelblue', label='Original')
axes[0].hist(X_aug[:,0], 20, alpha=0.6, color='tomato', label='Augmented')
axes[0].set(xlabel='Feature 0', title='Augmentation Distribution'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].plot(history, 'o-', color='steelblue', lw=2)
axes[1].set(xlabel='Round', ylabel='# Labeled', title='Self-Training Data Growth')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/synthetic_data.png')
print('Saved synthetic_data.png')